# 05. Interactive Visualizations with Plotly
## 📚 Learning Objectives

By completing this notebook, you will:
- Build interactive visualizations with Plotly
- Create interactive dashboards
- Add interactivity (hover, zoom, filter)
- Use Plotly Express for quick charts
- Share interactive visualizations as self-contained HTML files

## 🔗 Where this fits

**Builds on:** Course 05 — Unit 3, lesson 04 "Interactive Plotly Visualizations" — single charts become linked dashboards you can hand to someone else.

**Used later in:** Course 05 — Unit 5, lesson 09, where dashboards report on a model running in production.

---

This notebook covers practical activities from **Course 05, Unit 3**:
- Building interactive visualizations and dashboards with Plotly

---

## The Story: From Photos to Videos

Imagine you're showing photos (static charts). **Then** you create videos (interactive charts) - viewers can pause, zoom, rewind. **After** adding interactivity, viewers can explore on their own!

Same with visualization: **After** learning static charts, we add interactivity - users can hover for details, zoom to see patterns, filter to focus. **After** adding interactivity, charts become exploratory tools!

---

## Why Interactive Visualizations Matter

Interactive visualizations are essential because:
- **Exploration**: Users can explore data themselves
- **Engagement**: Interactive charts are more engaging
- **Insights**: Users discover insights through interaction
- **Professional**: Interactive dashboards impress stakeholders

**Common Student Questions:**
- **Q: When do I use Plotly vs Matplotlib?**
  - Answer: Matplotlib for static, Plotly for interactive
  - Example: Report PDF → matplotlib, Web dashboard → plotly
  - Rule: Static → matplotlib, Interactive/web → plotly
  
- **Q: What makes a chart interactive?**
  - Answer: Hover tooltips, zoom, pan, filter, click interactions
  - Example: Hover shows values, zoom focuses on area, filter changes data
  - Benefit: Users explore data dynamically

---

## Introduction

**Plotly** enables creation of interactive visualizations and dashboards. Unlike static charts, interactive visualizations allow users to explore data dynamically through hover, zoom, pan, and filter interactions.


## 🎯 The case: one HTML file, three billion page views

The Johns Hopkins CSSE COVID-19 dashboard (lesson 04's case) proved something specific
about this lesson's subject: **the deliverable of an analysis is often a link, not a
report**. Launched on 22 January 2020 by Lauren Gardner and Ensheng Dong, it peaked at
**3–4.5 billion requests a day** and had served **3.6 billion page views** by mid-2022,
because anybody with a browser could open it and find their own country. No install, no
account, no analyst in the loop.

That is exactly what `fig.write_html()` gives you at the end of this notebook: **one
self-contained file** that a colleague opens by double-clicking, keeps working offline, and
still hovers and zooms.

**What goes wrong without it.** You will build a chart below where every point is a
port-month carrying six facts — port, state, border, date, trucks, cars. A static scatter
shows two of the six. When your client points at a dot in the corner and asks "which port
is that?", the static answer is "let me re-run it and get back to you"; the interactive
answer is a tooltip. Multiply that by a fortnight of review meetings and you have the
business case for this lesson.


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Real data:** `border_crossing_data.csv` — the U.S. Bureau of Transportation
  Statistics record of every inbound crossing at every U.S. land border port,
  monthly from 1996 to March 2019 (346,733 rows). Columns: port, state, border
  (US-Canada / US-Mexico), month, measure (Trucks, Personal Vehicles,
  Pedestrians, …) and the count.
- plotly, pandas

**Outputs:** What you'll see when you run the cells

- Interactive scatter, line and bar charts of real border traffic
- A two-panel dashboard exported as one self-contained HTML file

---


In [1]:
# WHAT: Import NumPy, pandas, and both Plotly APIs.
# WHY: This notebook practices express for speed and graph_objects for control.

# Imports
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
# WHAT: Load the real U.S. border-crossing log and reshape it to one row per port-month.
# WHY: Interactive charts pay off when a point carries more than two numbers - hovering
#      here reveals the port, the month and three different traffic measures at once.

print("Part 1: Load the real border-crossing data")
print("-" * 70)

DATA_DIR = '../../../Course 04/datasets/raw/'

crossings = pd.read_csv(DATA_DIR + 'border_crossing_data.csv',
                        usecols=['Port Name', 'State', 'Border', 'Date', 'Measure', 'Value'])
crossings['Date'] = pd.to_datetime(crossings['Date'], format='%m/%d/%Y %I:%M:%S %p')

print(f"✓ Loaded {len(crossings):,} real border-crossing records "
      f"({crossings['Date'].min():%Y-%m} to {crossings['Date'].max():%Y-%m})")
print(f"  Borders: {crossings['Border'].value_counts().to_dict()}")

# Long -> wide: one row per (port, month) with a column per traffic measure.
MEASURES = ['Trucks', 'Personal Vehicles', 'Pedestrians']
df = (crossings[crossings['Measure'].isin(MEASURES)]
      .pivot_table(index=['Port Name', 'State', 'Border', 'Date'],
                   columns='Measure', values='Value', aggfunc='sum')
      .reset_index())
df.columns.name = None

# Keep port-months where all three measures were actually reported, from 2015 on.
df = df[(df['Date'] >= '2015-01-01')].dropna(subset=MEASURES)
df = df[df[MEASURES].gt(0).all(axis=1)]

print(f"\n✓ Reshaped to {len(df):,} port-months (2015 onward) with all three measures reported")
print(df.head().to_string(index=False))

Part 1: Load the real border-crossing data
----------------------------------------------------------------------
✓ Loaded 346,733 real border-crossing records (1996-01 to 2019-03)
  Borders: {'US-Canada Border': 266187, 'US-Mexico Border': 80546}

✓ Reshaped to 2,614 port-months (2015 onward) with all three measures reported
Port Name  State           Border       Date  Pedestrians  Personal Vehicles  Trucks
    Alcan Alaska US-Canada Border 2015-01-01          1.0              738.0   602.0
    Alcan Alaska US-Canada Border 2015-02-01         21.0              832.0   481.0
    Alcan Alaska US-Canada Border 2015-04-01         55.0             2742.0   629.0
    Alcan Alaska US-Canada Border 2015-05-01        200.0             6249.0   753.0
    Alcan Alaska US-Canada Border 2015-06-01        795.0             9871.0   659.0


In [3]:
# WHAT: Draw an interactive scatter of truck vs car crossings, one point per port-month.
# WHY: Hover turns a dense cloud into a browsable table - you can name the port behind any outlier.

# PART 2: INTERACTIVE SCATTER PLOT
# ============================================================================
print("\n" + "=" * 70)
print("PART 2: Interactive Scatter Plot")
print("=" * 70)

print("\n✅ Example 1: Basic Interactive Scatter Plot")
print("-" * 70)

# Do trucks and cars cross at the same ports? One point = one port in one month.
fig = px.scatter(df, x='Trucks', y='Personal Vehicles', color='Border',
                 size='Pedestrians', hover_data=['Port Name', 'State', 'Date'],
                 title='Truck vs car crossings per port-month (2015 onward)',
                 labels={'Trucks': 'Truck crossings', 'Personal Vehicles': 'Car crossings'})
fig.show()
print("💡 Try: Hover over points, zoom, pan, click legend to filter by border!")

corr = df['Trucks'].corr(df['Personal Vehicles'])
print(f"   Measured correlation between truck and car volume: {corr:.2f}")

# ============================================================================


PART 2: Interactive Scatter Plot

✅ Example 1: Basic Interactive Scatter Plot
----------------------------------------------------------------------


💡 Try: Hover over points, zoom, pan, click legend to filter by border!
   Measured correlation between truck and car volume: 0.68


In [4]:
# WHAT: Draw an interactive monthly time series with a range slider.
# WHY: 23 years of monthly data is unreadable at full zoom; the slider lets the reader pick the window.

# PART 3: INTERACTIVE LINE CHART
# ============================================================================
print("\n" + "=" * 70)
print("PART 3: Interactive Line Chart")
print("=" * 70)

print("\n✅ Example 2: Interactive Line Chart with Time Series")
print("-" * 70)

# Real monthly time series: total car crossings per border, 1996 to 2019.
ts_data = (crossings[crossings['Measure'] == 'Personal Vehicles']
           .groupby(['Date', 'Border'], as_index=False)['Value'].sum())

fig = px.line(ts_data, x='Date', y='Value', color='Border',
              title='Interactive Line Chart: monthly car crossings by border',
              labels={'Value': 'Car crossings', 'Date': 'Month'})
fig.update_xaxes(rangeslider_visible=True)  # Add range slider
fig.show()
print("💡 Try: Use range slider to zoom into 2001 or 2008, hover for values!")
print(f"   {len(ts_data):,} month-border points span "
      f"{ts_data['Date'].min():%Y-%m} to {ts_data['Date'].max():%Y-%m}")

# ============================================================================


PART 3: Interactive Line Chart

✅ Example 2: Interactive Line Chart with Time Series
----------------------------------------------------------------------


💡 Try: Use range slider to zoom into 2001 or 2008, hover for values!
   558 month-border points span 1996-01 to 2019-03


In [5]:
# WHAT: Aggregate to the ten busiest truck ports, then draw an interactive bar chart.
# WHY: Aggregate FIRST, plot second - a bar chart of raw rows would silently sum behind your back.

# PART 4: INTERACTIVE BAR CHART
# ============================================================================
print("\n" + "=" * 70)
print("PART 4: Interactive Bar Chart")
print("=" * 70)

print("\n✅ Example 3: Interactive Bar Chart")
print("-" * 70)

# Which ports carry the most trucks? Aggregate first, then plot.
category_data = (df.groupby('Port Name', as_index=False)['Trucks'].mean()
                   .sort_values('Trucks', ascending=False).head(10))
fig = px.bar(category_data, x='Port Name', y='Trucks',
             title='Interactive Bar Chart: top 10 ports by average monthly truck crossings',
             labels={'Trucks': 'Average trucks per month'},
             color='Trucks')
fig.show()
print("💡 Try: Hover for exact values, click legend items!")
print(category_data.to_string(index=False))

# ============================================================================


PART 4: Interactive Bar Chart

✅ Example 3: Interactive Bar Chart
----------------------------------------------------------------------


💡 Try: Hover for exact values, click legend items!
             Port Name        Trucks
                Laredo 180110.960784
               Detroit 137030.750000
 Buffalo-Niagara Falls  78923.549020
             Otay Mesa  75495.176471
               El Paso  64864.823529
               Hidalgo  50049.313725
         Calexico East  29883.254902
               Nogales  27982.392157
Champlain-Rouses Point  25665.039216
           Brownsville  19094.450980


## 💬 Discuss

Measured above on 346,733 real border records: trucks and cars correlate at **0.68** across
port-months; the busiest truck port by monthly average is **Laredo (180,111)**, well ahead
of Detroit (137,031); and the two-panel dashboard exported to **4,832 KB** of HTML.

1. Laredo averages 30% more trucks per month than Detroit, and the top ten span 180,111
   down to 19,094 — a factor of nine. Which of those two facts belongs in the *headline* of
   your dashboard, and which belongs in the tooltip? Justify by naming your reader.
2. Trucks and cars correlate at **0.68** here but at **0.98** in lesson 01 (vehicles vs
   their passengers). Explain the difference in one sentence each — what makes one pair
   near-identical and the other merely related?
3. 4.8 MB for two charts. Your client wants a dashboard with twelve. Estimate the file
   size, then decide: do you ship one big HTML file, twelve small ones, or something else
   entirely? What does each choice cost the reader?


## Part 5: A Mini Dashboard, Shared as HTML

Two charts side by side in one interactive figure — then exported to a single
self-contained HTML file. That file is how Plotly work is shared: anyone can
open it in a browser, no Python required.

In [6]:
# WHAT: Build a two-panel dashboard with make_subplots and export it as one self-contained HTML file.
# WHY: That single file is how Plotly work is shared - it needs a browser, not a Python install.

# PART 5: build a two-panel dashboard with make_subplots, then export it as ONE self-contained HTML file
print("\n" + "=" * 70)
print("PART 5: Mini Dashboard and HTML Export")
print("=" * 70)

from plotly.subplots import make_subplots
import os

dash = make_subplots(rows=1, cols=2,
                     subplot_titles=('Mean monthly trucks by border',
                                     'Trucks vs cars, per port-month'))
border_means = df.groupby('Border', as_index=False)['Trucks'].mean()
dash.add_trace(go.Bar(x=border_means['Border'], y=border_means['Trucks'],
                      name='mean trucks/month'), row=1, col=1)
dash.add_trace(go.Scatter(x=df['Trucks'], y=df['Personal Vehicles'], mode='markers',
                          name='port-months', text=df['Port Name']), row=1, col=2)
dash.update_layout(title_text='U.S. land border traffic (hover, zoom, pan all work)',
                   height=450)
dash.show()

# Sharing: write ONE self-contained HTML file
html_path = 'mini_dashboard.html'
dash.write_html(html_path)
size_kb = os.path.getsize(html_path) / 1024
print(f"\n✓ Dashboard exported to {html_path} ({size_kb:.0f} KB)")
print("  Open that file in any web browser - hover/zoom/pan still work.")
print("  This single HTML file is how Plotly visualizations are shared.")
print(border_means.to_string(index=False))


PART 5: Mini Dashboard and HTML Export



✓ Dashboard exported to mini_dashboard.html (4832 KB)
  Open that file in any web browser - hover/zoom/pan still work.
  This single HTML file is how Plotly visualizations are shared.
          Border       Trucks
US-Canada Border  6402.530044
US-Mexico Border 24500.301065


In [7]:
# WHAT: Recap the interactive features used (hover, zoom, legend toggling, selection) and summarize.
# WHY: Naming the interactions helps you consciously design for them in your own dashboards.

# PART 6: recap the interactive features you just used, then summarize the whole notebook
print("\n" + "=" * 70)
print("PART 6: Interactive Features")
print("=" * 70)

print("""
✅ Interactive Features Available:

1. Hover Tooltips
   - Show data values on hover (here: port name, state, month)
   - Customize tooltip content
   - Multiple columns in tooltip

2. Zoom and Pan
   - Zoom in/out with mouse wheel
   - Pan by dragging
   - Double-click to reset

3. Legend Interaction
   - Click legend items to show/hide (here: one border at a time)
   - Double-click to isolate

4. Selection
   - Box select to zoom
   - Lasso select for custom areas

5. Export
   - Download as PNG
   - Save as HTML
   - Share interactive charts
""")

# ============================================================================
# ============================================================================
print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print(f"""
✅ What you learned, using {len(crossings):,} real U.S. border-crossing records:
   1. Interactive Scatter: trucks vs cars per port-month, hover for the port
   2. Interactive Line: 1996-2019 monthly crossings with a range slider
   3. Interactive Bar: the ten busiest truck ports
   4. Interactive Features: Tooltips, zoom, pan, filter, select
   5. Mini Dashboard: Two charts in one figure with make_subplots
   6. Sharing: write_html() -> one self-contained file for any browser

🎯 Key Takeaways:
   - Plotly: Standard for interactive Python visualizations
   - Interactivity: Hover, zoom, pan, filter, select
   - Plotly Express: Quick interactive charts
   - Plotly Graph Objects: More control and customization

📚 Next Steps:
   - Example 06: Customizing and Annotating Visualizations
   - (Example 04 covered advanced Plotly features - dashboards, 3D)
""")
print("✅ Interactive visualization concepts understood!")


PART 6: Interactive Features

✅ Interactive Features Available:

1. Hover Tooltips
   - Show data values on hover (here: port name, state, month)
   - Customize tooltip content
   - Multiple columns in tooltip

2. Zoom and Pan
   - Zoom in/out with mouse wheel
   - Pan by dragging
   - Double-click to reset

3. Legend Interaction
   - Click legend items to show/hide (here: one border at a time)
   - Double-click to isolate

4. Selection
   - Box select to zoom
   - Lasso select for custom areas

5. Export
   - Download as PNG
   - Save as HTML
   - Share interactive charts


Summary

✅ What you learned, using 346,733 real U.S. border-crossing records:
   1. Interactive Scatter: trucks vs cars per port-month, hover for the port
   2. Interactive Line: 1996-2019 monthly crossings with a range slider
   3. Interactive Bar: the ten busiest truck ports
   4. Interactive Features: Tooltips, zoom, pan, filter, select
   5. Mini Dashboard: Two charts in one figure with make_subplots
   6. Sha

## ⚠️ Where this breaks

- **Self-contained HTML means the data ships with the chart.** The 4,832 KB file above
  contains every plotted value in plain text. That is wonderful for offline sharing and
  unacceptable if the underlying rows are confidential — a "chart" you email is a **data
  export**. Aggregate before plotting anything you would not attach as a CSV.
- **A dashboard is not a finding.** Four panels invite the reader to browse, which is not
  the same as telling them something. If you cannot write the one sentence the dashboard
  exists to support, the dashboard is decoration.
- **Interactivity does not survive the screenshot.** Assume your figure will end up as a
  static image in a slide. The default view must carry the message on its own.
- **The assumption that must hold: the aggregation behind the chart is the one you want.**
  The notebook aggregates to the ten busiest ports by *mean monthly trucks*. Rank by total
  volume, by growth, or by a different date window and the top ten change. A bar chart of
  raw rows would have summed silently — which is why the code aggregates first, on purpose.
- **Plotly's rendering is client-side and single-threaded.** Beyond roughly 50,000 points
  the browser stalls, and no amount of server capacity helps. This is a hard ceiling, not a
  tuning problem.
- **Cheaper alternative:** for a recurring internal report, a scheduled notebook that emails
  three annotated PNGs is read more often than a dashboard people must remember to visit.
  Build the dashboard when the reader's question genuinely varies — by region, by product,
  by date — and a fixed chart cannot anticipate it.


## 📚 References

1. Heer, J., & Shneiderman, B. (2012). *Interactive Dynamics for Visual Analysis*. ACM Queue, 10(2), 30-55. <https://doi.org/10.1145/2133416.2146416>
2. Bostock, M., Ogievetsky, V., & Heer, J. (2011). *D3: Data-Driven Documents*. IEEE Transactions on Visualization and Computer Graphics, 17(12), 2301-2309. <https://doi.org/10.1109/TVCG.2011.185>
3. Midway, S. R. (2020). *Principles of Effective Data Visualization*. Patterns, 1(9), 100141. <https://doi.org/10.1016/j.patter.2020.100141>